# Cyclistic Bike-Share Case Study

## How Do Annual Members and Casual Riders Differ?

---

### Business Task

Cyclistic, a Chicago-based bike-share company, wants to **maximize annual memberships** because annual members are significantly more profitable than casual riders. The marketing analytics team needs to understand **how casual riders and annual members use Cyclistic bikes differently** so that a targeted marketing strategy can be designed to convert casual riders into members.

### Data Source

This analysis uses **Divvy/Motivate International Inc. public trip data** covering **January through December 2025** (12 monthly CSV files, ~5.9 million rides). The data is made available under the [Divvy Data License Agreement](https://divvybikes.com/data-license-agreement). Note: rider personally identifiable information has been removed for privacy.

### Objective

Produce a comprehensive exploratory analysis that:
1. Cleans and validates the raw trip data
2. Engineers useful features (time-based, geographic, behavioral)
3. Compares member vs. casual rider behavior across multiple dimensions
4. Generates publication-quality visualizations
5. Delivers actionable recommendations backed by data

## Environment Setup

**Open in Google Colab:**  
If you are running this notebook in [Google Colab](https://colab.research.google.com/), you will need to upload the 12 CSV data files or mount your Google Drive.

To upload files directly:
```python
from google.colab import files
uploaded = files.upload()  # Use the file picker to select all 12 CSVs
```

To mount Google Drive (if your data is stored there):
```python
from google.colab import drive
drive.mount('/content/drive')
# Then set DATA_DIR to the appropriate path in your Drive
```

> **Note:** The `DATA_DIR` variable in the next cell controls where the notebook looks for CSV files. Update it to match your environment.

**Running locally:**  
Make sure you have Python 3.8+ with `pandas`, `matplotlib`, `seaborn`, and `numpy` installed.

In [ ]:
# Install dependencies (uncomment the line below if running in Colab)
# !pip install pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import os
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries imported successfully.')

---
## 1. Data Loading

We load all 12 monthly CSV files (January-December 2025) and concatenate them into a single DataFrame. Each file contains trip-level records with columns for ride ID, bike type, start/end timestamps, start/end station names and IDs, start/end coordinates, and member/casual classification.

In [ ]:
# ── CONFIGURATION ──
# Update DATA_DIR to point to the folder containing your 12 CSV files.
DATA_DIR = "."  # Current directory (change as needed)

# List of monthly data files (January-December 2025)
csv_files = [
    "202501-divvy-tripdata-2_2026Feb4.csv",  # Updated Jan file
    "202502-divvy-tripdata.csv",
    "202503-divvy-tripdata.csv",
    "202504-divvy-tripdata.csv",
    "202505-divvy-tripdata.csv",
    "202506-divvy-tripdata.csv",
    "202507-divvy-tripdata.csv",
    "202508-divvy-tripdata.csv",
    "202509-divvy-tripdata.csv",
    "202510-divvy-tripdata.csv",
    "202511-divvy-tripdata.csv",
    "202512-divvy-tripdata.csv",
]

# Load and concatenate all files
dfs = []
for f in csv_files:
    path = os.path.join(DATA_DIR, f)
    print(f"Loading {f}...")
    dfs.append(pd.read_csv(path, parse_dates=['started_at', 'ended_at']))

df = pd.concat(dfs, ignore_index=True)
initial_rows = len(df)

print(f"\n{'='*50}")
print(f"Total rows loaded: {initial_rows:,}")
print(f"Columns: {list(df.columns)}")
print(f"{'='*50}")
df.info()
df.head()

---
## 2. Data Cleaning

We apply the following cleaning steps:
1. **Remove duplicate rides** based on `ride_id`
2. **Compute ride duration** (`ride_length_min`) from start and end timestamps
3. **Remove invalid rides:**
   - Rides with zero or negative duration (data errors)
   - Rides shorter than 1 minute (likely false starts or re-docking)
   - Rides longer than 24 hours / 1,440 minutes (likely lost/stolen bikes or system errors)

In [ ]:
# ── DATA CLEANING ──
cleaning_log = []
cleaning_log.append(f"Initial rows: {initial_rows:,}")

# 2a. Remove duplicates on ride_id
before = len(df)
df = df.drop_duplicates(subset='ride_id')
after = len(df)
dupes = before - after
cleaning_log.append(f"Duplicates removed: {dupes:,} (remaining: {after:,})")
print(f"Duplicates removed: {dupes:,}")

# 2b. Compute ride duration in minutes
df['ride_length_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# 2c. Remove negative/zero duration rides
before = len(df)
df = df[df['ride_length_min'] > 0]
after = len(df)
removed = before - after
cleaning_log.append(f"Negative/zero duration rides removed: {removed:,} (remaining: {after:,})")
print(f"Negative/zero rides removed: {removed:,}")

# 2d. Remove rides < 1 minute (likely false starts)
before = len(df)
df = df[df['ride_length_min'] >= 1]
after = len(df)
removed = before - after
cleaning_log.append(f"Rides < 1 minute removed: {removed:,} (remaining: {after:,})")
print(f"Rides < 1 minute removed: {removed:,}")

# 2e. Remove rides > 24 hours (1440 minutes)
before = len(df)
df = df[df['ride_length_min'] <= 1440]
after = len(df)
removed = before - after
cleaning_log.append(f"Rides > 24 hours removed: {removed:,} (remaining: {after:,})")
print(f"Rides > 24 hours removed: {removed:,}")

# Summary
final_rows = len(df)
cleaning_log.append(f"Final clean rows: {final_rows:,}")
cleaning_log.append(f"Total rows removed: {initial_rows - final_rows:,} ({(initial_rows - final_rows) / initial_rows * 100:.1f}%)")

print(f"\n{'='*50}")
print(f"CLEANING SUMMARY")
print(f"{'='*50}")
for entry in cleaning_log:
    print(f"  {entry}")
print(f"{'='*50}")

---
## 3. Feature Engineering

We derive several new columns to support downstream analysis:

| Column | Description |
|--------|-------------|
| `day_of_week` | Day name (Monday-Sunday) |
| `month` | Month name (January-December) |
| `month_num` | Month number (1-12) |
| `hour` | Hour of ride start (0-23) |
| `season` | Winter, Spring, Summer, Fall |
| `is_weekend` | True if Saturday or Sunday |
| `is_round_trip` | True if start station == end station |

In [ ]:
# ── FEATURE ENGINEERING ──

# Ordering references
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
season_order = ['Winter', 'Spring', 'Summer', 'Fall']

# Time-based features
df['day_of_week'] = df['started_at'].dt.day_name()
df['month'] = df['started_at'].dt.month_name()
df['month_num'] = df['started_at'].dt.month
df['hour'] = df['started_at'].dt.hour
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# Season mapping
def get_season(m):
    if m in [12, 1, 2]:
        return 'Winter'
    elif m in [3, 4, 5]:
        return 'Spring'
    elif m in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month_num'].apply(get_season)

# Round trip detection (same start and end station)
df['is_round_trip'] = (
    df['start_station_name'].notna() &
    df['end_station_name'].notna() &
    (df['start_station_name'] == df['end_station_name'])
)

print("Feature engineering complete.")
print(f"New columns: day_of_week, month, month_num, hour, season, is_weekend, is_round_trip")
print(f"\nDataFrame shape: {df.shape}")
df.head(3)

---
## 4. Descriptive Statistics

Before creating visualizations, we compute key summary statistics that quantify the differences between member and casual riders.

In [ ]:
# ── DESCRIPTIVE STATISTICS ──
stats = {}

# --- Overall member vs. casual split ---
member_count = (df['member_casual'] == 'member').sum()
casual_count = (df['member_casual'] == 'casual').sum()
stats['total_rides'] = final_rows
stats['member_rides'] = int(member_count)
stats['casual_rides'] = int(casual_count)
stats['member_pct'] = round(member_count / final_rows * 100, 1)
stats['casual_pct'] = round(casual_count / final_rows * 100, 1)

print("RIDE VOLUME SPLIT")
print(f"  Members: {member_count:,} ({stats['member_pct']}%)")
print(f"  Casual:  {casual_count:,} ({stats['casual_pct']}%)")
print(f"  Total:   {final_rows:,}")

# --- Duration statistics ---
dur = df.groupby('member_casual')['ride_length_min'].agg(['mean', 'median'])
stats['member_avg_duration'] = round(dur.loc['member', 'mean'], 1)
stats['casual_avg_duration'] = round(dur.loc['casual', 'mean'], 1)
stats['member_median_duration'] = round(dur.loc['member', 'median'], 1)
stats['casual_median_duration'] = round(dur.loc['casual', 'median'], 1)
stats['duration_ratio'] = round(stats['casual_avg_duration'] / stats['member_avg_duration'], 1)

print(f"\nDURATION STATISTICS")
print(f"  Member  - Mean: {stats['member_avg_duration']} min | Median: {stats['member_median_duration']} min")
print(f"  Casual  - Mean: {stats['casual_avg_duration']} min | Median: {stats['casual_median_duration']} min")
print(f"  Casual/Member ratio: {stats['duration_ratio']}x")

# --- Bike type breakdown ---
bike_type = df.groupby(['member_casual', 'rideable_type']).size().unstack(fill_value=0)
for col in bike_type.columns:
    for idx in bike_type.index:
        stats[f'{idx}_{col}_count'] = int(bike_type.loc[idx, col])
        stats[f'{idx}_{col}_pct'] = round(bike_type.loc[idx, col] / bike_type.loc[idx].sum() * 100, 1)

print(f"\nBIKE TYPE BREAKDOWN")
print(bike_type)
print()
print((bike_type.div(bike_type.sum(axis=1), axis=0) * 100).round(1))

# --- Round trip percentage ---
rt = df.groupby('member_casual')['is_round_trip'].mean() * 100
stats['member_round_trip_pct'] = round(rt['member'], 1)
stats['casual_round_trip_pct'] = round(rt['casual'], 1)

print(f"\nROUND TRIP PERCENTAGE")
print(f"  Members: {stats['member_round_trip_pct']}%")
print(f"  Casual:  {stats['casual_round_trip_pct']}%")

# --- Weekend vs. weekday ---
weekend_data = df.groupby(['member_casual', 'is_weekend']).size().unstack(fill_value=0)
for idx in weekend_data.index:
    total = weekend_data.loc[idx].sum()
    stats[f'{idx}_weekend_pct'] = round(weekend_data.loc[idx][True] / total * 100, 1)
    stats[f'{idx}_weekday_pct'] = round(weekend_data.loc[idx][False] / total * 100, 1)

print(f"\nWEEKEND vs. WEEKDAY")
print(f"  Members - Weekday: {stats['member_weekday_pct']}% | Weekend: {stats['member_weekend_pct']}%")
print(f"  Casual  - Weekday: {stats['casual_weekday_pct']}% | Weekend: {stats['casual_weekend_pct']}%")

---
## 5. Visualizations

The following section generates 11 charts that illustrate the key behavioral differences between annual members and casual riders. Each chart is accompanied by a brief insight summary.

In [ ]:
# ── CHART STYLING ──

# Color palette
MEMBER_COLOR = "#2962A0"   # dark blue
CASUAL_COLOR = "#E67E22"   # coral/orange
COLORS = {"member": MEMBER_COLOR, "casual": CASUAL_COLOR}
BG_COLOR = "#FFFFFF"
GRID_COLOR = "#E8E8E8"

# Global matplotlib styling
plt.rcParams.update({
    'figure.facecolor': BG_COLOR,
    'axes.facecolor': BG_COLOR,
    'axes.grid': True,
    'grid.color': GRID_COLOR,
    'grid.alpha': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
})

def format_thousands(x, p):
    """Axis tick formatter: 1000 -> 1,000"""
    return f'{x:,.0f}'

print("Chart styling configured.")

### Chart 1: Ride Volume Split (Donut Chart)

Shows the overall proportion of rides taken by members vs. casual riders. Members account for the majority of ride volume, but casuals represent a meaningful conversion opportunity.

In [ ]:
# ── CHART 1: Donut chart - Overall split ──
fig, ax = plt.subplots(figsize=(7, 7))
sizes = [stats['member_rides'], stats['casual_rides']]
labels = [f"Members\n{stats['member_pct']}%", f"Casual\n{stats['casual_pct']}%"]
colors_list = [MEMBER_COLOR, CASUAL_COLOR]
wedges, texts = ax.pie(sizes, labels=labels, colors=colors_list,
                       startangle=90, wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))
for t in texts:
    t.set_fontsize(14)
    t.set_fontweight('bold')
ax.text(0, 0, f"{final_rows / 1e6:.1f}M\nTotal Rides", ha='center', va='center',
        fontsize=16, fontweight='bold', color='#333333')
ax.set_title("Members dominate ride volume, but casuals\nrepresent a significant conversion opportunity",
             fontsize=13, fontweight='bold', pad=20)
plt.show()

### Chart 2: Monthly Ride Trends

Both rider types follow a strong seasonal pattern peaking in summer, but casual ridership is **far more seasonal** -- their volume drops more dramatically in colder months.

In [ ]:
# ── CHART 2: Monthly ride trends ──
monthly = df.groupby(['month_num', 'month', 'member_casual']).size().reset_index(name='rides')
monthly_pivot = monthly.pivot_table(index=['month_num', 'month'], columns='member_casual',
                                     values='rides', fill_value=0).reset_index()
monthly_pivot = monthly_pivot.sort_values('month_num')

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_pivot['month'], monthly_pivot['member'], color=MEMBER_COLOR,
        marker='o', linewidth=2.5, markersize=6, label='Members', zorder=3)
ax.plot(monthly_pivot['month'], monthly_pivot['casual'], color=CASUAL_COLOR,
        marker='o', linewidth=2.5, markersize=6, label='Casual', zorder=3)
ax.fill_between(range(len(monthly_pivot)), monthly_pivot['member'].values, alpha=0.08, color=MEMBER_COLOR)
ax.fill_between(range(len(monthly_pivot)), monthly_pivot['casual'].values, alpha=0.08, color=CASUAL_COLOR)
ax.set_title("Both groups peak in summer, but casual ridership\nis far more seasonal than members", fontweight='bold')
ax.set_ylabel("Number of Rides")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_thousands))
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Chart 3: Rides by Day of Week

Members ride most heavily on weekdays (consistent with commuting), while casuals surge on Saturday and Sunday (consistent with leisure use).

In [ ]:
# ── CHART 3: Day of week rides ──
dow = df.groupby(['day_of_week', 'member_casual']).size().reset_index(name='rides')
dow_pivot = dow.pivot_table(index='day_of_week', columns='member_casual', values='rides').reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(day_order))
w = 0.35
ax.bar(x - w/2, dow_pivot['member'], w, color=MEMBER_COLOR, label='Members', zorder=3)
ax.bar(x + w/2, dow_pivot['casual'], w, color=CASUAL_COLOR, label='Casual', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(day_order, rotation=45, ha='right')
ax.set_ylabel("Number of Rides")
ax.set_title("Members ride consistently on weekdays (commuters);\ncasuals surge on weekends (leisure)", fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_thousands))
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
plt.tight_layout()
plt.show()

### Chart 4: Hourly Usage Pattern

Members display sharp commute peaks at 8 AM and 5 PM. Casuals build gradually through midday and peak in the afternoon, consistent with recreational use.

In [ ]:
# ── CHART 4: Hourly usage ──
hourly = df.groupby(['hour', 'member_casual']).size().reset_index(name='rides')
hourly_pivot = hourly.pivot_table(index='hour', columns='member_casual', values='rides', fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hourly_pivot.index, hourly_pivot['member'], color=MEMBER_COLOR,
        linewidth=2.5, label='Members', zorder=3)
ax.plot(hourly_pivot.index, hourly_pivot['casual'], color=CASUAL_COLOR,
        linewidth=2.5, label='Casual', zorder=3)
ax.fill_between(hourly_pivot.index, hourly_pivot['member'], alpha=0.08, color=MEMBER_COLOR)
ax.fill_between(hourly_pivot.index, hourly_pivot['casual'], alpha=0.08, color=CASUAL_COLOR)
ax.axvspan(7, 9, alpha=0.06, color=MEMBER_COLOR, label='_nolegend_')
ax.axvspan(16, 18, alpha=0.06, color=MEMBER_COLOR, label='_nolegend_')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h}:00' for h in range(24)], rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Number of Rides")
ax.set_title("Members show clear commute peaks (8 AM, 5 PM);\ncasuals build gradually through the afternoon", fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_thousands))
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
plt.tight_layout()
plt.show()

### Chart 5: Average Ride Duration

Casual riders take significantly longer rides on average, suggesting leisure-oriented trips rather than the efficient point-to-point commutes typical of members.

In [ ]:
# ── CHART 5: Average ride duration ──
fig, ax = plt.subplots(figsize=(7, 5))
dur_data = [stats['member_avg_duration'], stats['casual_avg_duration']]
bars = ax.bar(['Members', 'Casual'], dur_data, color=[MEMBER_COLOR, CASUAL_COLOR],
              width=0.5, zorder=3, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, dur_data):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f} min', ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_ylabel("Average Ride Duration (minutes)")
ax.set_title(f"Casual riders take {stats['duration_ratio']}x longer rides on average,\nsuggesting leisure-oriented usage", fontweight='bold')
ax.set_ylim(0, max(dur_data) * 1.25)
plt.tight_layout()
plt.show()

### Chart 6: Duration by Day of Week

Casual ride duration peaks on weekends, reinforcing the leisure pattern. Member duration remains remarkably consistent across all days, reflecting habitual commute behavior.

In [ ]:
# ── CHART 6: Duration by day of week ──
dur_dow = df.groupby(['day_of_week', 'member_casual'])['ride_length_min'].mean().reset_index()
dur_dow_pivot = dur_dow.pivot_table(index='day_of_week', columns='member_casual',
                                     values='ride_length_min').reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(day_order))
w = 0.35
ax.bar(x - w/2, dur_dow_pivot['member'], w, color=MEMBER_COLOR, label='Members', zorder=3)
ax.bar(x + w/2, dur_dow_pivot['casual'], w, color=CASUAL_COLOR, label='Casual', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(day_order, rotation=45, ha='right')
ax.set_ylabel("Average Ride Duration (minutes)")
ax.set_title("Casual ride duration peaks on weekends;\nmember duration stays remarkably consistent", fontweight='bold')
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
plt.tight_layout()
plt.show()

### Chart 7: Bike Type Preference

Compares the relative preference for classic bikes vs. electric bikes between the two rider groups.

In [ ]:
# ── CHART 7: Bike type preference ──
bt = df.groupby(['member_casual', 'rideable_type']).size().reset_index(name='count')
bt_pivot = bt.pivot_table(index='member_casual', columns='rideable_type', values='count', fill_value=0)
bt_pct = bt_pivot.div(bt_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bt_pct.index))
w = 0.35
cols = bt_pct.columns.tolist()
classic_color = '#3498DB'
electric_color = '#F39C12'
for i, col in enumerate(cols):
    c = classic_color if 'classic' in col else electric_color
    bars = ax.bar(x + i * w, bt_pct[col], w, label=col.replace('_', ' ').title(),
                  color=c, zorder=3, edgecolor='white', linewidth=1)
    for bar, val in zip(bars, bt_pct[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.0f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xticks(x + w / 2)
ax.set_xticklabels(['Casual', 'Members'], fontsize=12)
ax.set_ylabel("Percentage of Rides")
ax.set_title("Bike type preference by rider type", fontweight='bold')
ax.set_ylim(0, 100)
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
plt.tight_layout()
plt.show()

### Chart 8: Top 10 Start Stations for Casual Riders

Casual riders concentrate at tourist and lakefront locations (e.g., Streeter Dr & Grand Ave, DuSable Lake Shore), confirming their recreational usage pattern.

In [ ]:
# ── CHART 8: Top 10 stations for casual riders ──
casual_stations = (df[df['member_casual'] == 'casual']
                   .dropna(subset=['start_station_name'])
                   .groupby('start_station_name').size()
                   .nlargest(10)
                   .sort_values())

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(casual_stations.index, casual_stations.values, color=CASUAL_COLOR, zorder=3, height=0.6)
for bar, val in zip(bars, casual_stations.values):
    ax.text(val + casual_stations.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel("Number of Rides")
ax.set_title("Top 10 start stations for casual riders\n(concentrated at tourist & lakefront locations)", fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(format_thousands))
plt.tight_layout()
plt.show()

### Chart 9: Top 10 Start Stations for Members

Member rides originate from stations distributed across residential neighborhoods and business corridors, consistent with commuting behavior.

In [ ]:
# ── CHART 9: Top 10 stations for members ──
member_stations = (df[df['member_casual'] == 'member']
                   .dropna(subset=['start_station_name'])
                   .groupby('start_station_name').size()
                   .nlargest(10)
                   .sort_values())

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(member_stations.index, member_stations.values, color=MEMBER_COLOR, zorder=3, height=0.6)
for bar, val in zip(bars, member_stations.values):
    ax.text(val + member_stations.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel("Number of Rides")
ax.set_title("Top 10 start stations for members\n(distributed across residential & business corridors)", fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(format_thousands))
plt.tight_layout()
plt.show()

### Chart 10: Casual Riders' Share by Month

The proportion of rides taken by casual riders nearly doubles in summer months compared to winter, identifying the peak window for conversion marketing campaigns.

In [ ]:
# ── CHART 10: Seasonal casual share ──
seasonal = df.groupby(['month_num', 'month', 'member_casual']).size().reset_index(name='rides')
seasonal_pivot = seasonal.pivot_table(index=['month_num', 'month'], columns='member_casual',
                                       values='rides', fill_value=0).reset_index()
seasonal_pivot = seasonal_pivot.sort_values('month_num')
seasonal_pivot['total'] = seasonal_pivot['member'] + seasonal_pivot['casual']
seasonal_pivot['casual_share'] = seasonal_pivot['casual'] / seasonal_pivot['total'] * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(range(len(seasonal_pivot)), seasonal_pivot['casual_share'].values,
                alpha=0.3, color=CASUAL_COLOR, zorder=2)
ax.plot(range(len(seasonal_pivot)), seasonal_pivot['casual_share'].values,
        color=CASUAL_COLOR, linewidth=2.5, marker='o', markersize=6, zorder=3)
ax.set_xticks(range(len(seasonal_pivot)))
ax.set_xticklabels(seasonal_pivot['month'].values, rotation=45, ha='right')
ax.set_ylabel("Casual Rider Share (%)")
ax.set_title("Casual riders' share of total rides nearly doubles in summer,\nidentifying the peak conversion window", fontweight='bold')
ax.axhline(y=seasonal_pivot['casual_share'].mean(), color='#999999', linestyle='--', alpha=0.5, linewidth=1)
ax.text(11, seasonal_pivot['casual_share'].mean() + 0.5,
        f"Annual avg: {seasonal_pivot['casual_share'].mean():.0f}%",
        fontsize=10, color='#666666', ha='right')
for i, val in enumerate(seasonal_pivot['casual_share'].values):
    ax.text(i, val + 1, f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold', color=CASUAL_COLOR)
ax.set_ylim(0, seasonal_pivot['casual_share'].max() + 8)
plt.tight_layout()
plt.show()

### Chart 11: Weekday vs. Weekend Split

Members ride overwhelmingly on weekdays; casuals have a much more even split with significantly higher weekend preference.

In [ ]:
# ── CHART 11: Weekday vs Weekend split (horizontal stacked) ──
fig, ax = plt.subplots(figsize=(8, 4))
categories = ['Members', 'Casual']
weekday_vals = [stats['member_weekday_pct'], stats['casual_weekday_pct']]
weekend_vals = [stats['member_weekend_pct'], stats['casual_weekend_pct']]
y = np.arange(len(categories))
h = 0.4
ax.barh(y, weekday_vals, h, label='Weekday', color=MEMBER_COLOR, zorder=3)
ax.barh(y, weekend_vals, h, left=weekday_vals, label='Weekend', color=CASUAL_COLOR, zorder=3)
for i in range(len(categories)):
    ax.text(weekday_vals[i]/2, i, f'{weekday_vals[i]:.0f}%', ha='center', va='center',
            fontweight='bold', color='white', fontsize=12)
    ax.text(weekday_vals[i] + weekend_vals[i]/2, i, f'{weekend_vals[i]:.0f}%', ha='center', va='center',
            fontweight='bold', color='white', fontsize=12)
ax.set_yticks(y)
ax.set_yticklabels(categories, fontsize=12)
ax.set_xlabel("Percentage of Rides")
ax.set_title("Members are weekday-heavy; casuals\nshow much stronger weekend preference", fontweight='bold')
ax.legend(frameon=True, facecolor='white', edgecolor='#CCCCCC', fontsize=11)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

---
## 6. Key Findings

Based on the analysis above, here are the most important differences between annual members and casual riders:

1. **Ride Volume:** Members account for the majority of total rides, but casuals represent a large addressable conversion pool.

2. **Trip Duration:** Casual riders take significantly longer rides (roughly 2x the average duration of members), indicating leisure-oriented rather than utilitarian usage.

3. **Day-of-Week Pattern:** Members ride consistently on weekdays (commuter behavior); casuals surge on weekends (leisure behavior).

4. **Hourly Pattern:** Members show distinct AM/PM commute peaks (8 AM, 5 PM). Casuals build gradually through midday, peaking in the afternoon.

5. **Seasonality:** Both groups peak in summer, but casual ridership is far more seasonal. The casual share of total rides nearly doubles in warm months.

6. **Station Preferences:** Casuals concentrate at lakefront/tourist locations; members are distributed across residential and business corridors.

7. **Round Trips:** Casual riders take round trips at a higher rate, consistent with recreational loop rides.

8. **Bike Type:** Both groups use classic and electric bikes, with some variation in preference rates.

---
## 7. Recommendations

Based on the behavioral differences identified, here are three data-driven recommendations for converting casual riders into annual members:

### Recommendation 1: Launch a "Weekend Warrior" Membership Tier
**Data backing:** Casuals ride disproportionately on weekends, and their average ride duration is roughly 2x that of members. A weekend-only or limited membership at a reduced price point could capture riders who currently don't see value in a full annual pass because they don't commute by bike.

### Recommendation 2: Run Targeted Summer Conversion Campaigns at Top Casual Stations
**Data backing:** Casual rider share peaks in summer months, and their rides are concentrated at a small number of lakefront/tourist stations (e.g., Streeter Dr & Grand Ave). Placing physical signage, running in-app promotions, and offering limited-time membership discounts at these specific locations during June-August would reach the highest density of potential converts at peak engagement.

### Recommendation 3: Introduce a "Ride Longer, Save More" Value Proposition
**Data backing:** Casual riders take significantly longer trips on average, meaning they likely pay more per ride under the current pricing structure. Marketing materials should emphasize the per-ride cost savings of membership for riders who take trips longer than 15-20 minutes, with concrete "you would have saved $X" messaging in the app after each ride.

---
## 8. Methodology & Limitations

### Data Source
- **Provider:** Divvy / Motivate International Inc. (public trip data)
- **Period:** January 1, 2025 through December 31, 2025 (12 monthly files)
- **License:** [Divvy Data License Agreement](https://divvybikes.com/data-license-agreement)
- **Note:** The January file used (`202501-divvy-tripdata-2_2026Feb4.csv`) is a corrected/updated version.

### Cleaning Steps Applied
1. Removed duplicate rides (by `ride_id`)
2. Removed rides with zero or negative duration
3. Removed rides shorter than 1 minute (false starts / re-docking)
4. Removed rides longer than 24 hours (likely system errors or lost bikes)

### Known Limitations
- **No rider-level identification:** We cannot track individual users across rides or determine how many unique riders are in each group.
- **No pricing data:** We cannot calculate actual revenue impact or per-ride cost differences.
- **Station data gaps:** Some rides have missing start/end station names (likely from GPS-locked e-bikes not docked at stations), which limits geographic analysis.
- **No demographic data:** Rider age, gender, home ZIP code, etc. are not available due to privacy protections.
- **Single-year snapshot:** Trends may differ year-over-year; this analysis reflects 2025 only.
- **Correlation vs. causation:** Behavioral differences are observed patterns; they do not prove why riders choose one type over another.

---

*This notebook was created as part of the Google Data Analytics Professional Certificate capstone project. Analysis performed using Python (pandas, matplotlib, seaborn).*

*Data source: Divvy / Motivate International Inc. | Period: Jan-Dec 2025*